In [1]:
import gensim
import numpy as np
import pandas as pd
from ruwordnet import RuWordNet
from collections import Counter
from sklearn.neighbors import KDTree

In [2]:
from tqdm import notebook
from tqdm.auto import tqdm
tqdm.pandas()

In [3]:
from nltk.tokenize import word_tokenize

# Init WordNet

In [7]:
wn = RuWordNet()

# Load data

In [4]:
DF = pd.read_excel('база тематики лексики рки.xlsx')
LEMMA_LIST = DF['лемма с ударением'].values.tolist()

# Load FastText model (geowac-lemmas-skipgram)
download: http://vectors.nlpl.eu/repository/20/213.zip

In [4]:
original = 'FastText_Geowac/model.model'
big_model = gensim.models.fasttext.FastTextKeyedVectors.load(original)

# Func to get token vector 

In [5]:
def vectorize(text):
    text = str(text)
    vec = np.sum([big_model[word] for word in ' '.join(word_tokenize(text, language='russian')).lower().split()], axis=0)
    vec /= sum(vec**2) ** 0.5 
    return vec

# Vectorize and store synsets

In [8]:
words, vectors, synset_ids = [], [], []
for synset in notebook.tqdm(wn.synsets):
    # if synset.part_of_speech != 'V':
    #     continue
    for sense in synset.senses:
        words.append(sense.name)
        vectors.append(vectorize(sense.name))
        synset_ids.append(synset.id)

  0%|          | 0/59905 [00:00<?, ?it/s]

## Store synset_vectors in a searchable DB

In [9]:
vectors = np.stack(vectors)

In [10]:
tree = KDTree(vectors)

In [11]:
import pickle

In [12]:
with open('GeoWac_tokens_words_tree_synset_ids.pickle', 'wb') as handle:
    pickle.dump([words, tree, synset_ids], handle, protocol=pickle.HIGHEST_PROTOCOL)

# Load saved info from pickle

In [7]:
import pickle

In [ ]:
with open('GeoWac_tokens_words_tree_synset_ids.pickle', 'rb') as handle:
    words, tree, synset_ids = pickle.load(handle)

# Define distance measure and sense_retrieval function

In [9]:
def distance2vote(d, a=3, b=5):
    sim = np.maximum(0, 1 - d**2/2)
    return np.exp(-d**a) * sim **b

In [11]:
def get_WordNet_senses(token, top_n_senses=5):
    votes = Counter()
    dists, ids = tree.query(vectorize(token).reshape(1, -1), k=100)
    for idx, distance in zip(ids[0], dists[0]):
        for hyper in wn[synset_ids[idx]].hypernyms:
            for second_hyper in hyper.hypernyms:
                for third_hyper in second_hyper.hypernyms:
                    for fourth_hyper in third_hyper.hypernyms:
                        votes[fourth_hyper.id] += distance2vote(distance)
    out = []
    for sid, score in votes.most_common(top_n_senses):
        out.append([score, wn[sid].title])
    return out

# Examples

In [12]:
anatomy = 'анатомия'
eye = 'глаз'
nose = 'нос'
prow = 'нос самолета корабля'
mouth = 'рот'
credit_card = 'банковская карта'
engine = 'двигатель'
el_engine = 'электрический двигатель'

In [13]:
get_WordNet_senses(anatomy)

[[1.6848185748132936, 'СФЕРА ДЕЯТЕЛЬНОСТИ'],
 [1.0735164794104473, 'ПОСТОЯННАЯ СУЩНОСТЬ'],
 [0.5524306970459694, 'ПРОИСХОДЯЩАЯ СУЩНОСТЬ'],
 [0.4328819498550743, 'НАУКА'],
 [0.364748164792399, 'АБСТРАКТНАЯ СУЩНОСТЬ']]

In [14]:
get_WordNet_senses(eye)

[[1.3702968127108186, 'ФИЗИЧЕСКАЯ СУЩНОСТЬ'],
 [1.1232867789055432, 'ОБУСЛАВЛИВАТЬ, СПОСОБСТВОВАТЬ'],
 [1.006715689411066, 'ПОСТОЯННАЯ СУЩНОСТЬ'],
 [0.9546917984484333, 'ВЛИЯТЬ, ВОЗДЕЙСТВОВАТЬ'],
 [0.9540373230808104, 'НАРУШИТЬ СОСТОЯНИЕ, ХОД']]

In [15]:
get_WordNet_senses(nose)

[[3.263580056559362, 'ФИЗИЧЕСКАЯ СУЩНОСТЬ'],
 [1.8615208269526313, 'ПЕРЕЖИТЬ, ИСПЫТАТЬ'],
 [1.243255158537878, 'ДЕЙСТВИЕ, ЦЕЛЕНАПРАВЛЕННОЕ ДЕЙСТВИЕ'],
 [0.9920635756559333, 'ПОСТОЯННАЯ СУЩНОСТЬ'],
 [0.9307604134763158, 'ОТНОШЕНИЕ МЕЖДУ СУЩНОСТЯМИ']]

In [16]:
get_WordNet_senses(prow)

[[4.450932166628841, 'ТЕХНИЧЕСКОЕ УСТРОЙСТВО'],
 [3.4235483489359453, 'ПРИСПОСОБЛЕНИЕ (ПРЕДМЕТ)'],
 [2.2573903060845337, 'ФИЗИЧЕСКАЯ СУЩНОСТЬ'],
 [1.853356983308765, 'ФИЗИЧЕСКИЙ ОБЪЕКТ'],
 [1.5571941967951923, 'ВООРУЖЕНИЕ']]

In [17]:
get_WordNet_senses(mouth)

[[3.2913708919624067, 'ФИЗИЧЕСКАЯ СУЩНОСТЬ'],
 [2.5000544013822172, 'ПОСТОЯННАЯ СУЩНОСТЬ'],
 [0.6465606565442982, 'ОБУСЛАВЛИВАТЬ, СПОСОБСТВОВАТЬ'],
 [0.4370489083428027, 'ПРИНУДИТЬ, ЗАСТАВИТЬ'],
 [0.3603989462521738, 'ОТНОШЕНИЕ МЕЖДУ СУЩНОСТЯМИ']]

In [18]:
get_WordNet_senses(credit_card)

[[7.915670851161192, 'ТЕКСТ'],
 [5.544245305047735, 'ЗАПИСЬ (ТО, ЧТО ЗАПИСАНО)'],
 [4.735749527510243, 'ФИНАНСОВЫЙ ДОКУМЕНТ'],
 [4.000460509468343, 'ФИНАНСОВЫЙ ИНСТРУМЕНТ'],
 [3.4012123254632174, 'ДОКУМЕНТ']]

In [19]:
get_WordNet_senses(engine)

[[8.089134029616158, 'ПРИСПОСОБЛЕНИЕ (ПРЕДМЕТ)'],
 [7.293464286134525, 'ПРЕДМЕТ, ВЕЩЬ'],
 [6.850010509541247, 'ПРОМЫШЛЕННОЕ ОБОРУДОВАНИЕ'],
 [5.970307111766189, 'ЭНЕРГЕТИЧЕСКОЕ ОБОРУДОВАНИЕ'],
 [3.2177955378781053, 'ФИЗИЧЕСКИЙ ОБЪЕКТ']]

In [20]:
get_WordNet_senses(el_engine)

[[20.36949297365261, 'ПРИСПОСОБЛЕНИЕ (ПРЕДМЕТ)'],
 [12.806368503177987, 'ТЕХНИЧЕСКОЕ УСТРОЙСТВО'],
 [9.356482271391322, 'ПРЕДМЕТ, ВЕЩЬ'],
 [5.211554493684381, 'ОБОРУДОВАНИЕ'],
 [4.711535383629402, 'ПРОМЫШЛЕННОЕ ОБОРУДОВАНИЕ']]

# Annotate senses

In [21]:
DF['Значения'] = DF['лемма с ударением'].progress_apply(get_WordNet_senses)

  0%|          | 0/6149 [00:00<?, ?it/s]

In [22]:
DF['Значение1'] = DF['Значения'].progress_apply(lambda row: row[0][1])

  0%|          | 0/6149 [00:00<?, ?it/s]

In [23]:
DF['Значение2'] = DF['Значения'].progress_apply(lambda row: row[1][1])

  0%|          | 0/6149 [00:00<?, ?it/s]

In [24]:
DF['Значение3'] = DF['Значения'].progress_apply(lambda row: row[2][1] if len(row) >= 3 else None)

  0%|          | 0/6149 [00:00<?, ?it/s]

In [25]:
DF['Значение4'] = DF['Значения'].progress_apply(lambda row: row[3][1] if len(row) >= 4 else None)

  0%|          | 0/6149 [00:00<?, ?it/s]

In [26]:
DF['Значение5'] = DF['Значения'].progress_apply(lambda row: row[4][1] if len(row) >= 5 else None)

  0%|          | 0/6149 [00:00<?, ?it/s]

In [27]:
DF.head()

,Тема,лемма с ударением,грам. инфо,англ. перевод,уровень,Значения,Значение1,Значение2,Значение3,Значение4,Значение5
0,внешность,анатомия,только ед.ч.,anatomy,4,"[[1.6848185748132936, СФЕРА ДЕЯТЕЛЬНОСТИ], [1....",СФЕРА ДЕЯТЕЛЬНОСТИ,ПОСТОЯННАЯ СУЩНОСТЬ,ПРОИСХОДЯЩАЯ СУЩНОСТЬ,НАУКА,АБСТРАКТНАЯ СУЩНОСТЬ
1,внешность,блондин,ж.р. блондинка; р.п. мн.ч. блондинок,NaN,rki_no_level,"[[1.6386398955006283, ПОСТОЯННАЯ СУЩНОСТЬ], [1...",ПОСТОЯННАЯ СУЩНОСТЬ,БИОЛОГИЧЕСКАЯ СУЩНОСТЬ,"СВОЙСТВО, ХАРАКТЕРИСТИКА","СВОЙСТВО, ХАРАКТЕРИСТИКА",ЖИВОЙ ОРГАНИЗМ
2,внешность,бок,NaN,side,4,"[[1.5725032050957908, ПОСТОЯННАЯ СУЩНОСТЬ], [0...",ПОСТОЯННАЯ СУЩНОСТЬ,ФИЗИЧЕСКАЯ СУЩНОСТЬ,МЕСТО В ПРОСТРАНСТВЕ,"ПРИНУДИТЬ, ЗАСТАВИТЬ","ИЗМЕНИТЬ, СДЕЛАТЬ ИНЫМ"
3,внешность,борода,NaN,beard,3,"[[2.399233026807112, СЛОЙ (ПЛОСКАЯ ЧАСТЬ)], [0...",СЛОЙ (ПЛОСКАЯ ЧАСТЬ),"ПРЕДМЕТ, ВЕЩЬ",ФИЗИЧЕСКАЯ СУЩНОСТЬ,"ОБУСЛАВЛИВАТЬ, СПОСОБСТВОВАТЬ",ФИЗИЧЕСКИЙ ОБЪЕКТ
4,внешность,бородатый,(-ая; -ое; -ые),bearded,rki_no_level,"[[2.175330003778196, ЖИВОЙ ОРГАНИЗМ], [1.99868...",ЖИВОЙ ОРГАНИЗМ,СУБЪЕКТ ДЕЯТЕЛЬНОСТИ,ЖИВОЙ ОРГАНИЗМ,"СВОЙСТВО, ХАРАКТЕРИСТИКА",ЧЕЛОВЕК


In [28]:
DF.drop('Значения', axis=1, inplace=True)
DF.to_excel('база тематики лексики рки 4й гипероним.xlsx')

# Count senses

In [29]:
m1 = DF.Значение1.unique().tolist()
m2 = DF.Значение2.unique().tolist()
m3 = DF.Значение3.unique().tolist()
m4 = DF.Значение4.unique().tolist()
m5 = DF.Значение5.unique().tolist()

In [42]:
# print('SECOND_HYPER')
# len(m1), len(m2), len(m3), len(m4), len(m5)

SECOND_HYPER


(959, 1243, 1365, 1437, 1480)

In [63]:
# print('THIRD_HYPER')   
# len(m1), len(m2), len(m3), len(m4), len(m5)

THIRD_HYPER


(481, 667, 788, 835, 912)

In [30]:
print('FOURTH_HYPER')   
len(m1), len(m2), len(m3), len(m4), len(m5)

FOURTH_HYPER


(246, 365, 436, 490, 559)

In [31]:
union = set(m1).union(set(m2)).union(set(m3)).union(set(m4)).union(set(m5))
intersection = set(m1).intersection(set(m2)).intersection(set(m3)).intersection(set(m4)).intersection(set(m5))

In [43]:
# print('SECOND_HYPER')
# len(union), len(intersection)

SECOND_HYPER


(2713, 390)

In [65]:
# print('THIRD_HYPER')
# len(union), len(intersection)

THIRD_HYPER


(1413, 283)

In [36]:
print('FOURTH_HYPER')
len(union), len(intersection)

FOURTH_HYPER


(787, 168)

In [76]:
# print('THIRD_HYPER')
# sorted(m1)

THIRD_HYPER


['АБСТРАКТНАЯ СУЩНОСТЬ',
 'АВАРИЯ',
 'АВИАЦИОННАЯ ТЕХНИКА',
 'АВТОМОТОТРАНСПОРТНОЕ СРЕДСТВО',
 'АДМИНИСТРАТИВНО-ТЕРРИТОРИАЛЬНАЯ ЕДИНИЦА',
 'АКЦИЯ (ДЕЙСТВИЕ)',
 'АЛКОГОЛЬНЫЙ НАПИТОК',
 'АНТРОПОГЕННОЕ ВОЗДЕЙСТВИЕ',
 'АРТИСТ',
 'АУДИОВИЗУАЛЬНОЕ ПРОИЗВЕДЕНИЕ',
 'БАЛАНС (СООТНОШЕНИЕ)',
 'БЕДСТВИЕ',
 'БЕСПОЗВОНОЧНОЕ ЖИВОТНОЕ',
 'БИОЛОГИЧЕСКАЯ СУЩНОСТЬ',
 'БИОЛОГИЧЕСКИЙ ПРОЦЕСС',
 'БОЛЕЗНЬ',
 'БРОСИТЬСЯ (СТРЕМИТЕЛЬНО НАПРАВИТЬСЯ)',
 'БУМАЖНАЯ ПРОДУКЦИЯ',
 'ВАРИАНТ, РАЗНОВИДНОСТЬ',
 'ВВЕСТИ ВНУТРЬ',
 'ВЕЖЛИВЫЙ',
 'ВЕРУЮЩИЙ',
 'ВЕЩЕСТВО',
 'ВИД СПОРТА',
 'ВКЛЮЧИТЬ В СОСТАВ',
 'ВЛИЯТЬ, ВОЗДЕЙСТВОВАТЬ',
 'ВМЕСТИЛИЩЕ',
 'ВОБРАТЬ В СЕБЯ',
 'ВОДНОЕ ЖИВОТНОЕ',
 'ВОЕННОСЛУЖАЩИЙ',
 'ВОЗМЕЗДИЕ',
 'ВОИН (ЧЕЛОВЕК)',
 'ВОСПРОИЗВЕСТИ (ВОССОЗДАТЬ, ПОВТОРИТЬ В КОПИИ)',
 'ВРЕМЯ, ПРОДОЛЖИТЕЛЬНОСТЬ',
 'ВЫВОД, ЗАКЛЮЧЕНИЕ',
 'ВЫДЕЛЕНИЕ ВЕЩЕСТВА, ЭНЕРГИИ',
 'ВЫДЕЛИТЬ, ОТЛИЧИТЬ ОТ ОСТАЛЬНЫХ',
 'ВЫЗВАТЬ МЫСЛЬ, ЧУВСТВО',
 'ВЫЗВАТЬ, ПОРОДИТЬ',
 'ВЫПОЛНИТЬ, ИСПОЛНИТЬ, ОСУЩЕСТВИТЬ',
 'ВЫСКАЗАТЬ',
 'ВЫСКАЗАТЬ МНЕНИЕ',
 'В

In [35]:
print('FOURTH_HYPER')
sorted(m1)

FOURTH_HYPER


['АБСТРАКТНАЯ СУЩНОСТЬ',
 'АУДИОВИЗУАЛЬНОЕ ПРОИЗВЕДЕНИЕ',
 'БЕСПОЗВОНОЧНОЕ ЖИВОТНОЕ',
 'БИОЛОГИЧЕСКАЯ СУЩНОСТЬ',
 'БОЛЕЗНЬ',
 'ВЕЩЕСТВО',
 'ВЛИЯТЬ, ВОЗДЕЙСТВОВАТЬ',
 'ВМЕСТИЛИЩЕ',
 'ВОБРАТЬ В СЕБЯ',
 'ВОЗОБНОВЛЯЕМЫЕ ПРИРОДНЫЕ РЕСУРСЫ',
 'ВОИН (ЧЕЛОВЕК)',
 'ВРЕМЯ, МОМЕНТ ВРЕМЕНИ',
 'ВРЕМЯ, ПРОДОЛЖИТЕЛЬНОСТЬ',
 'ВЫДЕЛИТЬ ИЗ СОСТАВА',
 'ВЫЗВАТЬ, ПОРОДИТЬ',
 'ВЫПОЛНИТЬ, ИСПОЛНИТЬ, ОСУЩЕСТВИТЬ',
 'ВЫСКАЗАТЬ',
 'ВЫСКАЗЫВАНИЕ (ТО, ЧТО ВЫСКАЗАНО)',
 'ГАСТРОНОМИЯ',
 'ГОСУДАРСТВЕННЫЙ ОРГАН',
 'ГРУППА ЛЮДЕЙ',
 'ДАТЬ ВОЗМОЖНОСТЬ, ПОЗВОЛИТЬ',
 'ДВИГАТЬСЯ НА НОГАХ',
 'ДВИЖЕНИЕ, ПЕРЕМЕЩЕНИЕ',
 'ДЕЙСТВИЕ ЧЕЛОВЕКА',
 'ДЕЙСТВИЕ, ЦЕЛЕНАПРАВЛЕННОЕ ДЕЙСТВИЕ',
 'ДЕЛИТЬ, РАЗДЕЛЯТЬ НА ЧАСТИ',
 'ДЕЯТЕЛЬ',
 'ДОКУМЕНТ',
 'ДОМАШНЕЕ ИМУЩЕСТВО',
 'ДОМАШНИЙ СКОТ',
 'ДОСТАВИТЬ К МЕСТУ НАЗНАЧЕНИЯ',
 'ЕДИНИЦА ЯЗЫКА',
 'ЖИВОЙ ОРГАНИЗМ',
 'ЖИВОТНОЕ',
 'ЖИДКОСТЬ',
 'ЗАНЯТИЕ, ДЕЯТЕЛЬНОСТЬ',
 'ЗАПИСЬ (ТО, ЧТО ЗАПИСАНО)',
 'ЗВУК',
 'ЗНАК, ОБОЗНАЧЕНИЕ',
 'ЗОНА',
 'ИЗБАВИТЬСЯ',
 'ИЗДАВАТЬ, ИСПУСКАТЬ',
 'ИЗДЕЛИЕ',
 'ИЗДЕЛИЕ ЛЕГ

In [77]:
# print('THIRD_HYPER')
# intersection

THIRD_HYPER


{'АБСТРАКТНАЯ СУЩНОСТЬ',
 'АВТОМОТОТРАНСПОРТНОЕ СРЕДСТВО',
 'АДМИНИСТРАТИВНО-ТЕРРИТОРИАЛЬНАЯ ЕДИНИЦА',
 'АУДИОВИЗУАЛЬНОЕ ПРОИЗВЕДЕНИЕ',
 'БЕСПОЗВОНОЧНОЕ ЖИВОТНОЕ',
 'БИОЛОГИЧЕСКАЯ СУЩНОСТЬ',
 'БИОЛОГИЧЕСКИЙ ПРОЦЕСС',
 'БОЛЕЗНЬ',
 'ВАРИАНТ, РАЗНОВИДНОСТЬ',
 'ВЕРУЮЩИЙ',
 'ВЕЩЕСТВО',
 'ВИД СПОРТА',
 'ВЛИЯТЬ, ВОЗДЕЙСТВОВАТЬ',
 'ВМЕСТИЛИЩЕ',
 'ВОБРАТЬ В СЕБЯ',
 'ВОДНОЕ ЖИВОТНОЕ',
 'ВОСПРОИЗВЕСТИ (ВОССОЗДАТЬ, ПОВТОРИТЬ В КОПИИ)',
 'ВРЕМЯ, ПРОДОЛЖИТЕЛЬНОСТЬ',
 'ВЫДЕЛИТЬ, ОТЛИЧИТЬ ОТ ОСТАЛЬНЫХ',
 'ВЫЗВАТЬ МЫСЛЬ, ЧУВСТВО',
 'ВЫЗВАТЬ, ПОРОДИТЬ',
 'ВЫПОЛНИТЬ, ИСПОЛНИТЬ, ОСУЩЕСТВИТЬ',
 'ВЫСКАЗАТЬ',
 'ВЫСКАЗАТЬ МНЕНИЕ',
 'ВЫСКАЗЫВАНИЕ (ТО, ЧТО ВЫСКАЗАНО)',
 'ГАСТРОНОМИЯ',
 'ГЕОГРАФИЧЕСКИЙ ОБЪЕКТ',
 'ГОСУДАРСТВЕННАЯ ПОЛИТИКА',
 'ГОСУДАРСТВЕННЫЙ ОРГАН',
 'ГРУППА ЛЮДЕЙ',
 'ДАТЬ ВОЗМОЖНОСТЬ, ПОЗВОЛИТЬ',
 'ДВИЖЕНИЕ, ПЕРЕМЕЩЕНИЕ',
 'ДВИЖИМАЯ СОБСТВЕННОСТЬ',
 'ДЕЙСТВИЕ ЧЕЛОВЕКА',
 'ДЕЙСТВИЕ, ЦЕЛЕНАПРАВЛЕННОЕ ДЕЙСТВИЕ',
 'ДЕЙСТВИТЕЛЬНОЕ ЧИСЛО',
 'ДЕЛИТЬ, РАЗДЕЛЯТЬ НА ЧАСТИ',
 'ДЕЯТЕЛЬ',
 'ДЕЯТЕЛЬ ИСКУССТВА

In [34]:
print('FOURTH_HYPER')
sorted(intersection)

FOURTH_HYPER


['АБСТРАКТНАЯ СУЩНОСТЬ',
 'БЕСПОЗВОНОЧНОЕ ЖИВОТНОЕ',
 'БИОЛОГИЧЕСКАЯ СУЩНОСТЬ',
 'ВЕЩЕСТВО',
 'ВЛИЯТЬ, ВОЗДЕЙСТВОВАТЬ',
 'ВОБРАТЬ В СЕБЯ',
 'ВРЕМЯ, МОМЕНТ ВРЕМЕНИ',
 'ВРЕМЯ, ПРОДОЛЖИТЕЛЬНОСТЬ',
 'ВЫЗВАТЬ, ПОРОДИТЬ',
 'ВЫПОЛНИТЬ, ИСПОЛНИТЬ, ОСУЩЕСТВИТЬ',
 'ВЫСКАЗАТЬ',
 'ВЫСКАЗЫВАНИЕ (ТО, ЧТО ВЫСКАЗАНО)',
 'ГОСУДАРСТВЕННЫЙ ОРГАН',
 'ГРУППА ЛЮДЕЙ',
 'ДАТЬ ВОЗМОЖНОСТЬ, ПОЗВОЛИТЬ',
 'ДВИЖЕНИЕ, ПЕРЕМЕЩЕНИЕ',
 'ДЕЙСТВИЕ ЧЕЛОВЕКА',
 'ДЕЙСТВИЕ, ЦЕЛЕНАПРАВЛЕННОЕ ДЕЙСТВИЕ',
 'ДЕЛИТЬ, РАЗДЕЛЯТЬ НА ЧАСТИ',
 'ДОКУМЕНТ',
 'ДОМАШНЕЕ ИМУЩЕСТВО',
 'ДОСТАВИТЬ К МЕСТУ НАЗНАЧЕНИЯ',
 'ЕДИНИЦА ЯЗЫКА',
 'ЖИВОЙ ОРГАНИЗМ',
 'ЖИВОТНОЕ',
 'ЗАНЯТИЕ, ДЕЯТЕЛЬНОСТЬ',
 'ЗАПИСЬ (ТО, ЧТО ЗАПИСАНО)',
 'ЗНАК, ОБОЗНАЧЕНИЕ',
 'ИЗБАВИТЬСЯ',
 'ИЗДЕЛИЕ',
 'ИЗДЕЛИЕ ЛЕГКОЙ ПРОМЫШЛЕННОСТИ',
 'ИЗМЕНИТЬ, СДЕЛАТЬ ИНЫМ',
 'ИЗМЕНИТЬСЯ, ИЗМЕНЕНИЕ',
 'ИЗОБРАЖЕНИЕ (РЕЗУЛЬТАТ)',
 'ИМУЩЕСТВО, СОБСТВЕННОСТЬ',
 'ИНЖЕНЕРНОЕ ОБОРУДОВАНИЕ',
 'ИНФОРМАЦИЯ',
 'ИСКУССТВО',
 'ИСЧЕРПАЕМЫЕ ПРИРОДНЫЕ РЕСУРСЫ',
 'КАЛЕНДАРНАЯ ДАТА',
 'КАЧЕСТВО (СТЕПЕНЬ Ц